# Notebook 08 — Persistence Validation

**Issue #46 (P1-4).**

## 1. Context

The product is defined as persistently-bare, unregistered, developable land. Every element of that definition had been tested except the first. The pipeline had only ever been run against a single image date, and a single date cannot distinguish land that is bare because it is derelict from land that is bare because it was ploughed, stripped or resurfaced in the preceding weeks. Persistence was assumed to be the filter that separated the two.

The July 2026 labelling pilot had already established that the raw threshold detector produces land-use false positives at a high rate, returning nineteen non-sellable sites from nineteen inspected. That pilot was conducted on unfiltered output, however, and therefore measured the absence of land-use awareness rather than the quality of detection itself. The exclusion mask (P1-5) and the persistence filter (P1-4) were both introduced to address this, and the expectation was that the known false-positive classes would largely drop out once both were applied.

This notebook tests that expectation. Four seasonal scenes spanning fourteen months were processed, the candidates bare across all four were isolated, and every unregistered site in that set was inspected against aerial imagery. The hypothesis under test is that requiring bareness across four seasons removes transient bare ground and leaves a set enriched in genuine derelict land. It is falsifiable in an obvious way: if the persistent set is not so enriched, persistence is not the filter the product requires.

## 2. Method — scene selection

Four scenes were required, one per season, all Sentinel-2 L2A over Stoke-on-Trent. Selection could not be automated, and the reason is worth recording because it affects any future attempt to scale this process. The OData `cloudCover` attribute against which the pipeline filters is measured across the entire Sentinel-2 tile, an area of roughly 110 by 110 kilometres. Stoke occupies a small fraction of that footprint. A scene may therefore report eight per cent cloud tile-wide whilst a cloud bank sits directly over the city, or report twenty-five per cent whilst Stoke itself is clear beneath an otherwise cloudy tile. Filtering on the reported percentage selects the wrong scenes in both directions.

Each candidate date was consequently inspected by eye in true colour, judging cloud over the council area specifically and disregarding cloud elsewhere in the tile. The browser cloud filter was set permissively at seventy per cent so that scenes clear over Stoke but cloudy across the wider tile were not excluded from consideration before inspection.

| Season | Date | Notes |
|---|---|---|
| Summer | 2026-07-09 | Peak vegetation. Selected from three near-cloudless July candidates. |
| Autumn | 2025-09-22 | The only date in the September–October 2025 window clear over Stoke. |
| Winter | 2025-12-26 | Cloudiest season at this latitude; the least compromised scene available. |
| Spring | 2026-05-25 | Previously held scene, re-run under current code. |

The autumn date carries a known risk. September is post-harvest, when bare agricultural soil is at its most abundant and least distinguishable from brownfield. Ploughed fields entering the candidate pool on this date could produce false persistence, and this was recorded before the runs were conducted rather than offered afterwards as explanation.

A second constraint emerged during selection. Only one date in the entire September–October window was clear over Stoke, and the winter scene was accepted as the least compromised rather than the genuinely clear. Cloud cover at 53°N is the binding limitation on any multi-date approach in this region, and a wetter year would produce a materially different set.

### Database state prior to the runs

The `candidate_sites` table held 794 rows for the May 2026 date, accumulated across eight separate pipeline runs conducted over the FND, P0 and P1 development period. Their respective counts of 218, 218, 112, 81, 22 and 78 record successive changes to the detection algorithm rather than successive observations of the ground. Retaining them would have meant comparing measurements taken under different code against one another.

Those rows were archived to `candidate_sites_archive_20260723` and deleted. The predicate used was `std_bsi IS NULL`, which selects precisely the runs predating migration 004 and therefore constitutes an objective test of code generation rather than a judgement about dates. May 2026 was then re-run under current code and reproduced its previous figures exactly, at 90 candidates, 78 after exclusion and 18 register matches. That reproduction serves as an incidental verification that the merged pipeline is deterministic on a given scene.

All four dates analysed below therefore derive from a single run each, conducted under the same codebase.

## 3. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv

from src.database_query import get_db_connection

load_dotenv(ROOT / '.env')
conn = get_db_connection()

GSS = 'E06000021'
WINTER = '2025-12-26'   # anchor scene: strictest, fewest candidates
MATCH_M = 50            # centroid proximity threshold for cross-date matching

print('Connected to database')

Connected to database


## 4. Per-date results

Each row below represents one pipeline run against one scene. The counts are taken from the database rather than from the run logs, so they reflect what was actually persisted.

In [2]:
per_date = pd.read_sql("""
    SELECT image_date,
           COUNT(*)                                  AS candidates,
           COUNT(matched_site_reference)             AS register_matched,
           COUNT(*) - COUNT(matched_site_reference)  AS unregistered,
           ROUND(AVG(pixel_count)::numeric, 1)       AS mean_pixels,
           ROUND(AVG(bsi_value)::numeric, 4)         AS mean_bsi
    FROM candidate_sites
    WHERE gss_code = %(gss)s
    GROUP BY image_date
    ORDER BY image_date
""", conn, params={'gss': GSS})

per_date['register_recall_pct'] = (
    100 * per_date.register_matched / per_date.candidates
).round(1)

per_date

C:\Users\lward\AppData\Local\Temp\ipykernel_11036\750460860.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  per_date = pd.read_sql("""


,image_date,candidates,register_matched,unregistered,mean_pixels,mean_bsi,register_recall_pct
0,2025-09-22,153,31,122,11.5,0.1289,20.3
1,2025-12-26,72,11,61,8.8,0.1329,15.3
2,2026-05-25,78,18,60,9.7,0.1222,23.1
3,2026-07-09,251,41,210,12.7,0.1206,16.3


The candidate count varies substantially by season, from 72 in December to 251 in July, and the ordering is physically coherent. Bare-soil signal peaks in mid-summer and declines through autumn into winter as illumination falls, ground wets and vegetation senesces. The gate-passing pixel share recorded in the run logs follows the same progression, at 2.8 per cent in July, 1.7 per cent in September, 1.0 per cent in December and 0.8 per cent in May.

July exceeding May was not anticipated. The expectation had been that summer greening would suppress the count, since farmland reaches peak vegetation in July and should therefore fail the NDVI condition. The most plausible explanation is that drought-parched grassland and amenity land read as bare, which would make the summer scene a weaker discriminator than assumed rather than the stronger one. This bears on scene selection for any future multi-date work.

Register recall sits between 15.3 and 23.1 per cent across the four dates. The variation is not large but it is not negligible either, and it indicates that a proportion of the register is detectable only under particular seasonal conditions. More significantly, the ceiling is low in absolute terms. Notebook 05 established that register sites carry a mean NDVI of 0.21 and are therefore predominantly vegetated, whilst the detector gates on bare ground. Between three-quarters and four-fifths of the register is invisible to the pipeline by construction, before any question of threshold tuning arises. These figures form the baseline against which the persistent set is assessed in section 5.

## 5. Persistence

A candidate is treated as persistent where a candidate from every other date lies within 50 metres of it. The winter scene is used as the anchor for this comparison. It is the strictest of the four, having produced only 72 candidates against July's 251, and a site that is bare in late December as well as across the other three seasons represents the strongest persistence signal that four scenes can supply.

The stored `prior_date_count` column is not used here. That feature is computed at insert time and counts only those dates already present in the table when the row was written, which means it reflects the order in which the runs happened to be executed rather than true cross-date persistence. The query below is symmetric and therefore order-independent.

In [3]:
persistence = pd.read_sql("""
    SELECT dates_present, COUNT(*) AS sites FROM (
      SELECT w.id,
        (SELECT COUNT(DISTINCT o.image_date)
           FROM candidate_sites o
          WHERE o.gss_code = w.gss_code
            AND ST_DWithin(
                  ST_SetSRID(ST_MakePoint(o.utm_x, o.utm_y), 32630),
                  ST_SetSRID(ST_MakePoint(w.utm_x, w.utm_y), 32630),
                  %(m)s)
        ) AS dates_present
      FROM candidate_sites w
      WHERE w.gss_code = %(gss)s AND w.image_date = %(winter)s
    ) t
    GROUP BY dates_present
    ORDER BY dates_present
""", conn, params={'gss': GSS, 'winter': WINTER, 'm': MATCH_M})

persistence

C:\Users\lward\AppData\Local\Temp\ipykernel_11036\788387734.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  persistence = pd.read_sql("""


,dates_present,sites
0,1,26
1,2,11
2,3,15
3,4,20


Of the 72 winter candidates, 26 appear on the winter date alone. These are transient bare ground — wet ground, temporary works, a single ploughing — and persistence discards them, which is the behaviour the filter was built to produce. A further 11 recur on one other date and 15 on two others. Twenty are bare across all four scenes.

That set of twenty is what the product definition describes. The remainder of this notebook examines what is actually in it.

One methodological limitation should be stated before the results are read. Candidates are matched between dates by centroid proximity within 50 metres, which is a coarse test. A large irregular site whose centroid shifts between dates could fail to match itself, and two distinct adjacent sites could match one another. Candidate geometry is now persisted following FND-3, so a polygon-overlap join would be tighter and should replace this approach before any figure derived from it is published externally. The counts below are therefore approximately rather than exactly correct.

**Correction (following Notebook 09).** The recall figures used throughout this notebook are artefacts of the matching method rather than measurements of detection. Register sites were credited as detected where a candidate fell within 100 metres of the recorded site location, and that radius is wide enough to capture candidates with no relationship to the site. Tightening it reduces matches from 105 sites to 45 at 50 metres and 16 at 25 metres, a rate of collapse consistent with coincidental proximity, and the median distance from a register site to the nearest candidate is 154 metres. Notebook 09 further established that no register site satisfies the gate condition of BSI above 0.1 and NDVI below 0.2, the highest value observed anywhere in the register being 0.0843. Recall against the register is structurally zero.

The argument in this notebook survives the correction and is strengthened by it. The collapse from single-date recall to persistent-set recall was computed on a consistent basis, so the relative movement holds even though the absolute figures do not. What changes is the interpretation. Persistence was not degrading a detector which worked moderately well; it was operating on a candidate pool which had no correspondence to the register at all. The finding that persistence selects for permanence of use rather than absence of it remains correct, and the prior cause is now established: nothing in the candidate pool could ever have been registered brownfield, so the pool necessarily consisted of something else.

In [4]:
persistent = pd.read_sql("""
    SELECT w.id,
           w.utm_x::int  AS utm_x,
           w.utm_y::int  AS utm_y,
           w.pixel_count,
           ROUND((w.pixel_count * 0.04)::numeric, 2) AS hectares,
           ROUND(w.bsi_value::numeric, 4)            AS bsi,
           ROUND(w.compactness::numeric, 3)          AS compactness,
           w.matched_site_reference
    FROM candidate_sites w
    WHERE w.gss_code = %(gss)s AND w.image_date = %(winter)s
      AND (SELECT COUNT(DISTINCT o.image_date)
             FROM candidate_sites o
            WHERE o.gss_code = w.gss_code
              AND ST_DWithin(
                    ST_SetSRID(ST_MakePoint(o.utm_x, o.utm_y), 32630),
                    ST_SetSRID(ST_MakePoint(w.utm_x, w.utm_y), 32630),
                    %(m)s)) = 4
    ORDER BY w.pixel_count DESC
""", conn, params={'gss': GSS, 'winter': WINTER, 'm': MATCH_M})

n_registered = persistent.matched_site_reference.notna().sum()
print(f'persistent sites: {len(persistent)}  |  '
      f'register-matched: {n_registered}  |  '
      f'unregistered: {len(persistent) - n_registered}')

persistent

persistent sites: 20  |  register-matched: 1  |  unregistered: 19


C:\Users\lward\AppData\Local\Temp\ipykernel_11036\2301055334.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  persistent = pd.read_sql("""


,id,utm_x,utm_y,pixel_count,hectares,bsi,compactness,matched_site_reference
0,1362,552814,5872881,36,1.44,0.1323,0.032,NaN
1,1361,558332,5873071,15,0.60,0.1355,0.082,NaN
2,1320,553623,5880895,15,0.60,0.1739,0.184,NaN
3,1367,559253,5872372,15,0.60,0.1748,0.097,NaN
4,1371,557177,5871791,13,0.52,0.1204,0.113,NaN
5,1319,555165,5881132,12,0.48,0.1197,0.094,NaN
6,1350,554682,5874976,11,0.44,0.1277,0.154,NaN
7,1331,552827,5878471,9,0.36,0.1500,0.087,NaN
8,1364,555196,5872602,9,0.36,0.1176,0.283,NaN
9,1375,555260,5871260,9,0.36,0.1207,0.126,NaN


Two features of this table warrant attention before any site is inspected.

The first is register recall. Single dates matched the register at between 15.3 and 23.1 per cent of candidates. The persistent set matches at one site in twenty, or five per cent. If the register is accepted as a reasonable proxy for genuine brownfield, then persistence has moved the candidate pool away from brownfield rather than towards it. That is the opposite of the intended effect and it was visible in the data before validation began.

The second is compactness. The measure used here is the Polsby–Popper ratio, four pi A over P squared, for which a circle scores 1.0 and a square 0.785. The persistent set ranges from 0.032 to 0.321, meaning every member is long and thin. The largest site in the set, at 1.44 hectares, scores 0.032 and is therefore a ribbon rather than a parcel. Sites of that shape are characteristic of linear infrastructure — railway corridors, road verges, canal banks, service yards — rather than of developable plots, which tend towards compact boundaries because they were laid out to be built upon.

Neither signal is conclusive on its own. A demolished terrace plot or a narrow infill site can also score low on compactness, and the register is an imperfect proxy given that councils register what they choose to register. Taken together, however, both indicators suggested before inspection that the persistent set was unlikely to contain the target.

## 6. Manual validation

Each of the nineteen unregistered persistent candidates was inspected individually against aerial imagery and labelled according to a controlled vocabulary. The label `sellable` was reserved for a discrete parcel, with identifiable boundaries, showing no active use, that a developer could plausibly acquire. Everything else received a false-positive class describing what the site actually turned out to be.

Labels are read from CSV rather than embedded in this notebook, so the analysis can be re-run as further sites are labelled without editing code.

One limitation applies throughout. Google's aerial imagery may predate the satellite scenes used here, and in one case the discrepancy could not be resolved. Where a site's status was ambiguous because of that mismatch it was recorded with the uncertainty noted rather than guessed at.

In [5]:
labels = pd.read_csv(ROOT / 'data' / 'groundtruth' / 'persistent_labels.csv')
labelled = labels[labels.label.notna() & (labels.label.astype(str).str.strip() != '')]

print(f'labelled: {len(labelled)} of {len(labels)} unregistered persistent candidates')

breakdown = (labelled.label.value_counts()
             .rename_axis('label')
             .reset_index(name='sites'))

sellable = (labelled.label == 'sellable').sum()
print(f'sellable: {sellable} / {len(labelled)}  =  '
      f'precision {100 * sellable / len(labelled):.1f}%')

breakdown

labelled: 19 of 19 unregistered persistent candidates
sellable: 0 / 19  =  precision 0.0%


,label,sites
0,active_industrial,13
1,active_institutional,2
2,heritage_constrained,1
3,active_retail,1
4,car_park,1
5,construction,1


In [6]:
table = labelled[['candidate_id', 'hectares', 'compactness', 'label', 'site_name']].copy()
table = table.sort_values('hectares', ascending=False)
table

,candidate_id,hectares,compactness,label,site_name
0,1362,1.44,0.032,active_institutional,Royal Stoke University Hospital
1,1361,0.60,0.082,active_industrial,"Mossfield Road estate - D&G Bus, IAE"
2,1320,0.60,0.184,active_institutional,Ormiston Horizon Academy
3,1367,0.60,0.097,active_industrial,Park Hall Business Village
4,1371,0.52,0.113,active_industrial,Fenton - Victoria Industrial Complex
5,1319,0.48,0.094,heritage_constrained,Chatterley Whitfield Colliery - scheduled monu...
6,1350,0.44,0.154,active_retail,Holdcroft car dealership - stock yards and acc...
7,1375,0.36,0.126,car_park,"bet365 Stadium car park, Sideway"
8,1364,0.36,0.283,active_industrial,"Hyde Park Industrial Estate, Whieldon Road"
9,1331,0.36,0.087,active_industrial,"Daniel Platts Business Park, Charles Clowes Dr..."


Every inspected candidate resolved to a site in active use, under construction, or subject to designation preventing development. Thirteen were active industrial premises, two were institutional, one retail, one a surface car park, one an active construction site, and one a scheduled monument.

In every case the detected footprint consists of roofs and hardstanding rather than land. Warehouse and unit roofs, bus and lorry yards, hospital and school service areas, and surface parking account for the entire set. Two recurring signatures are worth noting. The first is the north-light sawtooth roof, a building type characteristic of Potteries factories, which appeared at both Fen Park and Grove Road. The second is the distribution loading apron, the open concrete between units which is kept clear because vehicles manoeuvre across it, and which is therefore not a building at all. That distinction has consequences for the design of any subsequent filter, since a mask built from building footprints alone would not remove it.

Three sites warrant individual comment because each points towards a different corrective measure.

Chatterley Whitfield Colliery is genuinely derelict brownfield and the detector was correct to identify it. It is also a scheduled monument on the Heritage at Risk register and cannot be cleared and developed. It is therefore not a sellable lead, but neither is it a detection failure. It represents a third category the pipeline has no concept of: land which is neither in active use nor available for development.

The bet365 Stadium car park should have been removed by the exclusion filter. The class `car_park` is already treated as a hard exclusion, and the site survived only because it is absent from the `exclusion_zones` table. This is a data coverage gap rather than a logic error, and it implies the remaining hard classes may be similarly incomplete.

The Haywood Hospital site is under active construction in current aerial imagery, but that imagery is undated and it could not be established whether work began before or after the scene window of September 2025 to July 2026. It is recorded as construction with the uncertainty noted rather than resolved.

This result is consistent with the July 2026 pilot conducted on single-date unfiltered output, which also returned no sellable sites from nineteen inspected. Persistence did not alter the character of the false positives. It selected for a particular subtype of them.

## 7. Finding

The hypothesis is not supported. Requiring bareness across four seasons does remove transient bare ground, and twenty-six of the seventy-two winter candidates were discarded on that basis, but the set which survives is dominated by active commercial and institutional sites rather than derelict land.

The mechanism is straightforward once stated. Persistence selects for permanence of use, not for absence of it. A warehouse roof, a bus depot yard and a hospital car park are bare in every season precisely because they are in continuous use and are maintained in that condition. Derelict land behaves in the opposite manner. It is colonised by vegetation within a season or two of abandonment, and may therefore read as vegetated on one or more dates, at which point the filter excludes it. The test does not merely fail to find the target; under some conditions it actively selects against it.

The root cause lies upstream of persistence. BSI and NDVI separate vegetation from non-vegetation, which is the task they were designed for. Neither separates bare ground from hard surface, and a tarmac yard is close to indistinguishable from a demolished plot in both indices. Persistence then compounds the error, because hard surfaces are the most reliably non-vegetated features in an urban area and will therefore recur across every date by construction.

Notebook 04 anticipated part of this. It recorded that threshold-only detection would identify currently-bare land and little else, and that the multi-band approach had been abandoned. That warning was borne out by the low register recall observed on single dates. What this notebook establishes is that the limitation is not corrected by the addition of further dates. The problem is not temporal sampling; it is the choice of spectral discriminant, and beneath that, the assumption that a legal and economic status can be inferred from surface reflectance at a single point in time.

### Caveats

These are stated so that the conclusion is not read as stronger than the evidence supports.

The cross-date join is coarse. Candidates are matched between dates by centroid proximity within fifty metres. A large irregular site whose centroid shifts between dates could fail to match itself, and two distinct adjacent sites could match one another. Candidate geometry is now persisted following FND-3, so a polygon-overlap join would be tighter and should replace the present approach before any figure derived from it is published externally.

The threshold used here is stricter than the ticket specifies. Issue #46 defines the persistence criterion as k of N with a suggested default of three of four. This analysis used all four, which is the strictest available reading. Fifteen candidates sit at three of four and remain unlabelled. Those sites are bare in three seasons and vegetated in one, which is closer to the expected signature of land left undisturbed long enough for vegetation to establish and then die back. Until they are inspected, the k of N criterion is untested and the finding above applies only to the all-of-four case.

The size threshold is uncalibrated. `min_pixels = 5`, equivalent to 0.2 hectares at twenty metre resolution, ships as a provisional value pending issue #89. The composition of the candidate pool is therefore provisional with it, and a different threshold would produce a different set.

Four dates is a small sample. Only one usable scene was found per season, and in the autumn window only one date in two months was clear over Stoke at all. A wetter or drier year would produce a different set of scenes and potentially a different result.

The register is an imperfect proxy. Recall against it is used throughout as an indicator of whether the detector is finding brownfield, but councils register the sites they choose to register, and the register skews towards larger and better-documented sites. A five per cent recall figure is informative but should not be treated as authoritative.

## 8. What follows

Three corrective measures are indicated by the evidence above, and they are distinct from one another.

The first is a filter for active land use, tracked as issue #96. The nineteen sites inspected here supply the worked examples. The scope of that ticket requires revision, however, because building-footprint subtraction alone would not be sufficient. Several of the detected footprints are loading aprons, service roads and turning circles between units rather than the units themselves. These are open hardstanding, not buildings, and they are kept clear by continuous use. A filter built from building outlines would leave them in place. Industrial land-use polygons, or building footprints buffered outward to their curtilage, would be required instead.

The second is an audit of exclusion zone coverage, tracked as issue #100. The bet365 Stadium car park reached the persistent set despite `car_park` already being a hard exclusion class, because that site is absent from the `exclusion_zones` table. If a site of that prominence is missing then the remaining classes are likely to be similarly incomplete, and the filter is being evaluated on data that does not represent what it was designed to remove.

The third is a designation overlay, tracked as issue #101. Chatterley Whitfield demonstrates that correct detection and commercial viability are separate questions. Historic England publish scheduled monuments and listed buildings as open data, and a candidate falling within a designated area should be flagged rather than silently discarded, since the detection itself was sound.

Beyond these, the finding bears directly on the architecture decision left open in Notebook 07. That notebook identified two possible designs: fork A, in which a classifier re-ranks candidates emitted by the threshold gate, and fork B, in which a learned model replaces the gate entirely and is trained on register locations sampled independently of any threshold. Notebook 07 noted that fork A is mathematically capped near the gate's own recall. This notebook establishes something further, which is that the candidate pool the gate produces is not merely small but is composed almost entirely of the wrong class of object. A classifier trained on that pool cannot surface land which never entered it. The evidence therefore favours fork B, or at minimum requires that the gate be corrected before any model is trained on its output.

The immediate sequence is to characterise the register itself, tracked as issue #104. Every notebook to date has examined the detector's output. None has examined its target. Until the spectral and geometric distribution of known brownfield is established, there is no basis on which to select features for a replacement detector, and no way to determine whether such a detector is feasible at ten metre resolution.

In [7]:
conn.close()
print('Connection closed')

Connection closed
